# Modelo Predictivo de Mortalidad Hospitalaria — GRD Chile 2024

**Objetivo:** Predecir mortalidad intrahospitalaria (TIPOALTA == 'FALLECIDO') a partir de variables clínicas y administrativas del sistema GRD del sector público chileno.

**Estructura del notebook:**
1. Carga y exploración inicial
2. Feature engineering
3. División y balanceo de clases
4. Modelos (con y sin `GRD_MORTALIDAD_SCORE`)
5. Evaluación y comparación
6. Exportación de resultados

**Nota metodológica — Data leakage:**  
El campo `IR_29301_MORTALIDAD` (riesgo de mortalidad del GRD) se asigna **al momento del alta**, no al ingreso. Incluirlo en el modelo equivale a usar información del futuro para predecir el presente. Se entrenan dos versiones del modelo: con y sin esta variable, para cuantificar su impacto y documentar el riesgo de *leakage*.

## 0. Setup e imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print('imbalanced-learn disponible — se usará SMOTE')
except ImportError:
    SMOTE_AVAILABLE = False
    print('imbalanced-learn no disponible — se usará class_weight=balanced')

try:
    from category_encoders import TargetEncoder
    TARGET_ENC_AVAILABLE = True
    print('category_encoders disponible — se usará TargetEncoder para COD_HOSPITAL')
except ImportError:
    TARGET_ENC_AVAILABLE = False
    print('category_encoders no disponible — se usará frecuencia para COD_HOSPITAL')

# Directorios de salida
PLOTS_DIR = Path('../plots')
MODELS_DIR = Path('../models')
DATA_PROC_DIR = Path('../data/processed')
for d in [PLOTS_DIR, MODELS_DIR, DATA_PROC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print('Setup completo.')

imbalanced-learn no disponible — se usará class_weight=balanced
category_encoders no disponible — se usará frecuencia para COD_HOSPITAL
Setup completo.


## 1. Carga de datos

El archivo GRD tiene un header duplicado en la fila 0 del CSV (fila 1 del archivo).  
Se lee con `header=0` y se elimina la primera fila si contiene los mismos nombres de columna.

In [2]:
DATA_PATH = '../data/archivosDuros/GRD_PUBLICO_2024.txt'

print(f'Leyendo {DATA_PATH} ...')
df_raw = pd.read_csv(
    DATA_PATH,
    sep='|',
    encoding='latin1',
    dtype=str,          # leer todo como string para mayor robustez
    low_memory=False
)
print(f'Shape original: {df_raw.shape}')

# Eliminar fila duplicada de headers (fila 0 del dataframe)
first_row = df_raw.iloc[0]
if (first_row == df_raw.columns).mean() > 0.5:
    print('Fila de headers duplicados detectada — eliminando...')
    df_raw = df_raw.iloc[1:].reset_index(drop=True)
    print(f'Shape tras eliminar fila duplicada: {df_raw.shape}')

df = df_raw.copy()
print('\nColumnas disponibles:')
print(list(df.columns))

Leyendo ../data/archivosDuros/GRD_PUBLICO_2024.txt ...
Shape original: (1085813, 129)

Columnas disponibles:
['COD_HOSPITAL', 'ID_BENEFICIARIO', 'SEXO', 'FECHA_NACIMIENTO', 'ETNIA', 'PROVINCIA', 'COMUNA', 'NACIONALIDAD', 'PREVISION', 'SERVICIO_SALUD', 'TIPO_PROCEDENCIA', 'TIPO_INGRESO', 'ESPECIALIDAD_MEDICA', 'TIPO_ACTIVIDAD', 'FECHA_INGRESO', 'SERVICIOINGRESO', 'FECHATRASLADO1', 'SERVICIOTRASLADO1', 'FECHATRASLADO2', 'SERVICIOTRASLADO2', 'FECHATRASLADO3', 'SERVICIOTRASLADO3', 'FECHATRASLADO4', 'SERVICIOTRASLADO4', 'FECHATRASLADO5', 'SERVICIOTRASLADO5', 'FECHATRASLADO6', 'SERVICIOTRASLADO6', 'FECHATRASLADO7', 'SERVICIOTRASLADO7', 'FECHATRASLADO8', 'SERVICIOTRASLADO8', 'FECHATRASLADO9', 'SERVICIOTRASLADO9', 'FECHAALTA', 'SERVICIOALTA', 'TIPOALTA', 'CONDICIONDEALTANEONATO1', 'PESORN1', 'SEXORN1', 'RN1ESTADO', 'CONDICIONDEALTANEONATO2', 'PESORN2', 'SEXORN2', 'RN2ESTADO', 'CONDICIONDEALTANEONATO3', 'PESORN3', 'SEXORN3', 'RN3ESTADO', 'CONDICIONDEALTANEONATO4', 'PESORN4', 'SEXORN4', 'RN4ESTA

In [3]:
# Vista rápida
df.head(3)

,COD_HOSPITAL,ID_BENEFICIARIO,SEXO,FECHA_NACIMIENTO,ETNIA,PROVINCIA,COMUNA,NACIONALIDAD,PREVISION,SERVICIO_SALUD,...,FECHAPROCEDIMIENTO1,FECHAINTERV1,ESPECIALIDADINTERVENCION,MEDICOALTA_ENCRIPTADO,USOSPABELLON,IR_29301_COD_GRD,IR_29301_PESO,IR_29301_SEVERIDAD,IR_29301_MORTALIDAD,HOSPPROCEDENCIA
0,107102,77447810,MUJER,1939-09-18,NINGUNO,MARGA MARGA,VILLA ALEMANA,CHILE,FONASA INSTITUCIONAL - (MAI) B,VIÑA DEL MAR QUILLOTA,...,NaN,2024-03-23,OFTALMOLOGÍA,71043147,1,022360,"0,4384",0,0,NaN
1,107100,81269680,MUJER,1952-05-31,NINGUNO,VALPARAISO,VIÑA DEL MAR,CHILE,FONASA INSTITUCIONAL - (MAI) D,VIÑA DEL MAR QUILLOTA,...,NaN,NaN,NaN,78588090,NaN,041023,"5,8207",3,3,NaN
2,105101,77590791,HOMBRE,2012-07-17,NINGUNO,ELQUI,LA SERENA,CHILE,FONASA INSTITUCIONAL - (MAI) A,COQUIMBO,...,NaN,2024-01-26,"CIRUGÍA DE CABEZA, CUELLO Y MAXILOFACIAL",75735052,2,034141,"0,4462",1,1,HOSPITAL SAN PABLO (COQUIMBO)


In [4]:
# Distribución de TIPOALTA
print('Distribución TIPOALTA:')
print(df['TIPOALTA'].value_counts(dropna=False))

Distribución TIPOALTA:
TIPOALTA
DOMICILIO                                        967852
HOSPITALIZACIÓN DOMICILIARIA                      36732
FALLECIDO                                         26682
DERIVACIÓN OTRO HOSPITAL DEL SERVICIO             24923
ALTA VOLUNTARIA                                   10651
DERIVACIÓN OTRO HOSPITAL DE LA RED NACIONAL        7938
DERIVACIÓN A OTROS CENTROS (CÁRCEL, HOGAR DE       3745
FUGA DEL PACIENTE                                  3299
DERIVACIÓN INST. PRIVADA (COMPRA DE SERVICIOS      2865
DERIVACIÓN INST. PRIVADA (VOLUNTARIO)              1126
Name: count, dtype: int64


## 2. Variable objetivo

`FALLECIDO = 1` si TIPOALTA es 'FALLECIDO', `0` en cualquier otro caso.

In [5]:
df['FALLECIDO'] = (df['TIPOALTA'].str.strip().str.upper() == 'FALLECIDO').astype(int)

n_total   = len(df)
n_fallec  = df['FALLECIDO'].sum()
pct       = 100 * n_fallec / n_total
print(f'Total registros  : {n_total:,}')
print(f'Fallecidos       : {n_fallec:,} ({pct:.2f}%)')
print(f'No fallecidos    : {n_total - n_fallec:,} ({100 - pct:.2f}%)')

Total registros  : 1,085,813
Fallecidos       : 26,682 (2.46%)
No fallecidos    : 1,059,131 (97.54%)


## 3. Feature Engineering

Se construyen 13 variables predictoras a partir de los campos crudos del GRD.

### 3.1 EDAD — años al momento del ingreso

In [6]:
def parse_date_col(series):
    """Parsear fechas YYYY-MM-DD con coerción de errores."""
    return pd.to_datetime(series.str.strip(), format='%Y-%m-%d', errors='coerce')

df['_FECHA_NAC']  = parse_date_col(df['FECHA_NACIMIENTO'])
df['_FECHA_ING']  = parse_date_col(df['FECHA_INGRESO'])
df['_FECHA_ALTA'] = parse_date_col(df['FECHAALTA'])

# Edad en años
df['EDAD'] = ((df['_FECHA_ING'] - df['_FECHA_NAC']).dt.days / 365.25).round(1)

# Sanear valores imposibles
df.loc[df['EDAD'] < 0, 'EDAD'] = np.nan
df.loc[df['EDAD'] > 120, 'EDAD'] = np.nan

# Imputar con mediana
mediana_edad = df['EDAD'].median()
n_nulos_edad = df['EDAD'].isna().sum()
df['EDAD'] = df['EDAD'].fillna(mediana_edad)
print(f'EDAD — nulos imputados: {n_nulos_edad:,} con mediana {mediana_edad:.1f} años')
print(df['EDAD'].describe())

EDAD — nulos imputados: 55 con mediana 47.1 años
count    1.085813e+06
mean     4.605497e+01
std      2.550968e+01
min      0.000000e+00
25%      2.660000e+01
50%      4.710000e+01
75%      6.780000e+01
max      1.089000e+02
Name: EDAD, dtype: float64


### 3.2 SEXO_BIN — binarización del sexo

In [7]:
df['_SEXO'] = df['SEXO'].str.strip().str.upper()
print('Valores SEXO antes de filtrar:', df['_SEXO'].value_counts(dropna=False).to_dict())

# Eliminar DESCONOCIDO
n_antes = len(df)
df = df[df['_SEXO'].isin(['HOMBRE', 'MUJER'])].copy()
n_despues = len(df)
print(f'Registros eliminados (SEXO DESCONOCIDO): {n_antes - n_despues:,}')

df['SEXO_BIN'] = (df['_SEXO'] == 'MUJER').astype(int)
print('SEXO_BIN — distribución:', df['SEXO_BIN'].value_counts().to_dict())

Valores SEXO antes de filtrar: {'MUJER': 632623, 'HOMBRE': 453074, 'DESCONOCIDO': 116}
Registros eliminados (SEXO DESCONOCIDO): 116
SEXO_BIN — distribución: {1: 632623, 0: 453074}


### 3.3 DIAS_HOSPITALIZACION

In [8]:
df['DIAS_HOSPITALIZACION'] = (df['_FECHA_ALTA'] - df['_FECHA_ING']).dt.days
# Truncar negativos a 0, imputar nulos con 0
df['DIAS_HOSPITALIZACION'] = df['DIAS_HOSPITALIZACION'].clip(lower=0).fillna(0).astype(int)
print('DIAS_HOSPITALIZACION:')
print(df['DIAS_HOSPITALIZACION'].describe())

DIAS_HOSPITALIZACION:
count    1.085697e+06
mean     5.687169e+00
std      1.212145e+01
min      0.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      6.000000e+00
max      6.610000e+02
Name: DIAS_HOSPITALIZACION, dtype: float64


### 3.4 N_TRASLADOS_INTERNOS, N_DIAGNOSTICOS, N_PROCEDIMIENTOS

Se cuentan los campos no nulos por fila en cada grupo de columnas.

In [9]:
def count_notnull_prefix(df, prefix, n_max):
    """Contar campos no nulos/vacíos para columnas prefix1..prefixN."""
    cols = [c for c in [f'{prefix}{i}' for i in range(1, n_max + 1)] if c in df.columns]
    if not cols:
        return pd.Series(0, index=df.index)
    return df[cols].apply(lambda c: c.str.strip().replace('', np.nan)).notna().sum(axis=1)

df['N_TRASLADOS_INTERNOS'] = count_notnull_prefix(df, 'FECHATRASLADO', 9)
df['N_DIAGNOSTICOS']       = count_notnull_prefix(df, 'DIAGNOSTICO', 35)
df['N_PROCEDIMIENTOS']     = count_notnull_prefix(df, 'PROCEDIMIENTO', 30)

for col in ['N_TRASLADOS_INTERNOS', 'N_DIAGNOSTICOS', 'N_PROCEDIMIENTOS']:
    print(f'{col}: min={df[col].min()}, max={df[col].max()}, media={df[col].mean():.2f}')

N_TRASLADOS_INTERNOS: min=0, max=9, media=0.24
N_DIAGNOSTICOS: min=1, max=35, media=5.78
N_PROCEDIMIENTOS: min=0, max=30, media=8.02


### 3.5 GRD_SEVERIDAD y GRD_MORTALIDAD_SCORE

Variables ordinales del sistema GRD (0–3).  
Los registros con valor 0 no tienen GRD asignado; se mantienen como categoría 0.

In [10]:
def to_ordinal_grd(series):
    """Convertir a entero 0-3, nulos → 0."""
    return pd.to_numeric(series.str.strip(), errors='coerce').fillna(0).astype(int).clip(0, 3)

# Intentar nombres con y sin IR_29301_ prefijo
sev_col  = next((c for c in df.columns if 'SEVERIDAD' in c.upper()), None)
mort_col = next((c for c in df.columns if 'MORTALIDAD' in c.upper() and c.upper() != 'FALLECIDO'), None)
peso_col = next((c for c in df.columns if 'PESO' in c.upper() and '29301' in c.upper()), None)

print(f'Columna severidad   : {sev_col}')
print(f'Columna mortalidad  : {mort_col}')
print(f'Columna peso GRD    : {peso_col}')

df['GRD_SEVERIDAD'] = to_ordinal_grd(df[sev_col]) if sev_col else 0
df['GRD_MORTALIDAD_SCORE'] = to_ordinal_grd(df[mort_col]) if mort_col else 0

print('\nGRD_SEVERIDAD:', df['GRD_SEVERIDAD'].value_counts().sort_index().to_dict())
print('GRD_MORTALIDAD_SCORE:', df['GRD_MORTALIDAD_SCORE'].value_counts().sort_index().to_dict())

Columna severidad   : IR_29301_SEVERIDAD
Columna mortalidad  : IR_29301_MORTALIDAD
Columna peso GRD    : IR_29301_PESO

GRD_SEVERIDAD: {0: 212054, 1: 387230, 2: 268808, 3: 217605}
GRD_MORTALIDAD_SCORE: {0: 212054, 1: 509388, 2: 180059, 3: 184196}


### 3.6 GRD_PESO

In [11]:
if peso_col:
    df['GRD_PESO'] = pd.to_numeric(
        df[peso_col].str.strip().str.replace(',', '.', regex=False),
        errors='coerce'
    ).fillna(0.0)
else:
    df['GRD_PESO'] = 0.0

print('GRD_PESO:')
print(df['GRD_PESO'].describe())

GRD_PESO:
count    1.085697e+06
mean     9.666301e-01
std      1.087385e+00
min      0.000000e+00
25%      4.804000e-01
50%      6.915000e-01
75%      1.040000e+00
max      2.064610e+01
Name: GRD_PESO, dtype: float64


### 3.7 TIPO_INGRESO — one-hot encoding

Categorías esperadas: URGENCIA, PROGRAMADA, OBSTETRICA.

In [12]:
tipo_ing_col = next((c for c in df.columns if 'TIPO_INGRESO' in c.upper() or 'TIPOINGRESO' in c.upper()), None)
print(f'Columna tipo ingreso: {tipo_ing_col}')

if tipo_ing_col:
    df['_TIPO_INGRESO'] = df[tipo_ing_col].str.strip().str.upper()
    print('Valores:', df['_TIPO_INGRESO'].value_counts(dropna=False).to_dict())
    tipo_dummies = pd.get_dummies(df['_TIPO_INGRESO'], prefix='TIPO_INGRESO', dtype=int)
    df = pd.concat([df, tipo_dummies], axis=1)
    TIPO_ING_COLS = list(tipo_dummies.columns)
else:
    TIPO_ING_COLS = []
    print('No se encontró columna TIPO_INGRESO')

print('Columnas creadas:', TIPO_ING_COLS)

Columna tipo ingreso: TIPO_INGRESO
Valores: {'URGENCIA': 552037, 'PROGRAMADA': 388688, 'OBSTETRICA': 144929, 'DESCONOCIDO': 43}
Columnas creadas: ['TIPO_INGRESO_DESCONOCIDO', 'TIPO_INGRESO_OBSTETRICA', 'TIPO_INGRESO_PROGRAMADA', 'TIPO_INGRESO_URGENCIA']


### 3.8 PREVISION_SIMPLIF — simplificación y one-hot encoding

In [13]:
prev_col = next((c for c in df.columns if 'PREVISION' in c.upper()), None)
print(f'Columna previsión: {prev_col}')

def simplificar_prevision(val):
    if pd.isna(val):
        return 'OTRO'
    v = str(val).strip().upper()
    if 'FONASA' in v and ('MAI' in v or 'TRAMO' in v or 'A ' in v or 'B ' in v or 'C ' in v or 'D ' in v):
        return 'FONASA_MAI'
    if 'FONASA' in v and 'LIBRE' in v:
        return 'FONASA_LIBRE'
    if 'FONASA' in v:
        return 'FONASA_MAI'  # default para FONASA sin especificar
    if 'ISAPRE' in v:
        return 'ISAPRE'
    return 'OTRO'

if prev_col:
    df['PREVISION_SIMPLIF'] = df[prev_col].apply(simplificar_prevision)
    print('PREVISION_SIMPLIF:', df['PREVISION_SIMPLIF'].value_counts().to_dict())
    prev_dummies = pd.get_dummies(df['PREVISION_SIMPLIF'], prefix='PREV', dtype=int)
    df = pd.concat([df, prev_dummies], axis=1)
    PREV_COLS = list(prev_dummies.columns)
else:
    PREV_COLS = []

print('Columnas creadas:', PREV_COLS)

Columna previsión: PREVISION
PREVISION_SIMPLIF: {'FONASA_MAI': 1072379, 'OTRO': 7503, 'ISAPRE': 5815}
Columnas creadas: ['PREV_FONASA_MAI', 'PREV_ISAPRE', 'PREV_OTRO']


### 3.9 ESPECIALIDAD_TOP — top 15 especialidades como dummies

In [14]:
esp_col = next((c for c in df.columns if 'ESPECIALIDAD' in c.upper()), None)
print(f'Columna especialidad: {esp_col}')

if esp_col:
    df['_ESP'] = df[esp_col].str.strip().str.upper().fillna('OTRA')
    top15 = df['_ESP'].value_counts().head(15).index.tolist()
    df['ESPECIALIDAD_TOP'] = df['_ESP'].where(df['_ESP'].isin(top15), other='OTRA')
    esp_dummies = pd.get_dummies(df['ESPECIALIDAD_TOP'], prefix='ESP', dtype=int)
    df = pd.concat([df, esp_dummies], axis=1)
    ESP_COLS = list(esp_dummies.columns)
    print(f'Top 15 especialidades capturan {df["_ESP"].isin(top15).mean()*100:.1f}% de los registros')
else:
    ESP_COLS = []

print('Columnas creadas:', len(ESP_COLS))

Columna especialidad: ESPECIALIDAD_MEDICA
Top 15 especialidades capturan 89.3% de los registros
Columnas creadas: 16


### 3.10 COD_HOSPITAL — Target Encoding

Con ~30–60 hospitales posibles, one-hot encoding sería ineficiente.  
Se usa **Target Encoding** (media de la variable objetivo por hospital) cuando `category_encoders` está disponible,  
y **Frequency Encoding** como fallback.

In [15]:
hosp_col = next((c for c in df.columns if 'COD_HOSPITAL' in c.upper() or ('HOSPITAL' in c.upper() and 'COD' in c.upper())), None)
print(f'Columna hospital: {hosp_col}')

if hosp_col:
    df['_HOSP'] = df[hosp_col].str.strip().fillna('DESCONOCIDO')
    n_hospitales = df['_HOSP'].nunique()
    print(f'Hospitales únicos: {n_hospitales}')

    if TARGET_ENC_AVAILABLE:
        # Target encoding se aplicará por fold durante CV para evitar leakage
        # Aquí calculamos versión global para uso en evaluación final
        te = TargetEncoder(cols=['_HOSP'], smoothing=10)
        df['HOSP_TARGET_ENC'] = te.fit_transform(df[['_HOSP']], df['FALLECIDO'])['_HOSP']
        HOSP_COL_FINAL = 'HOSP_TARGET_ENC'
    else:
        # Frequency encoding
        freq_map = df['_HOSP'].value_counts(normalize=True)
        df['HOSP_FREQ_ENC'] = df['_HOSP'].map(freq_map)
        HOSP_COL_FINAL = 'HOSP_FREQ_ENC'

    print(f'Columna hospital final: {HOSP_COL_FINAL}')
    print(df[HOSP_COL_FINAL].describe())
else:
    HOSP_COL_FINAL = None
    print('No se encontró columna COD_HOSPITAL')

Columna hospital: COD_HOSPITAL
Hospitales únicos: 72
Columna hospital final: HOSP_FREQ_ENC
count    1.085697e+06
mean     1.952658e-02
std      9.587114e-03
min      2.951100e-03
25%      1.216822e-02
50%      1.927610e-02
75%      2.498856e-02
max      4.551086e-02
Name: HOSP_FREQ_ENC, dtype: float64


## 4. Preparación del dataset final

Se definen dos conjuntos de features:
- **Sin leakage** (`X_sin`): excluye `GRD_MORTALIDAD_SCORE`
- **Con leakage** (`X_con`): incluye `GRD_MORTALIDAD_SCORE` (solo para referencia)

In [16]:
BASE_FEATURES = [
    'EDAD', 'SEXO_BIN', 'DIAS_HOSPITALIZACION',
    'N_TRASLADOS_INTERNOS', 'N_DIAGNOSTICOS', 'N_PROCEDIMIENTOS',
    'GRD_SEVERIDAD', 'GRD_PESO',
]

if HOSP_COL_FINAL:
    BASE_FEATURES.append(HOSP_COL_FINAL)

ALL_FEATURES_SIN  = BASE_FEATURES + TIPO_ING_COLS + PREV_COLS + ESP_COLS
ALL_FEATURES_CON  = ALL_FEATURES_SIN + ['GRD_MORTALIDAD_SCORE']

# Verificar que todas las columnas existen
ALL_FEATURES_SIN = [c for c in ALL_FEATURES_SIN if c in df.columns]
ALL_FEATURES_CON = [c for c in ALL_FEATURES_CON if c in df.columns]

print(f'Features sin leakage : {len(ALL_FEATURES_SIN)}')
print(f'Features con leakage : {len(ALL_FEATURES_CON)}')

# Dataset limpio: eliminar filas con NaN en features
df_model = df[ALL_FEATURES_CON + ['FALLECIDO']].copy()
df_model = df_model.apply(pd.to_numeric, errors='coerce')

n_antes = len(df_model)
df_model = df_model.dropna()
n_despues = len(df_model)
print(f'Filas eliminadas por NaN: {n_antes - n_despues:,}')
print(f'Dataset final: {df_model.shape}')
print(f'Tasa de mortalidad: {df_model["FALLECIDO"].mean()*100:.2f}%')

Features sin leakage : 32
Features con leakage : 33
Filas eliminadas por NaN: 0
Dataset final: (1085697, 34)
Tasa de mortalidad: 2.46%


In [17]:
y = df_model['FALLECIDO']
X_sin = df_model[ALL_FEATURES_SIN]
X_con = df_model[ALL_FEATURES_CON]

# Split estratificado 80/20
X_sin_tr, X_sin_te, y_tr, y_te = train_test_split(
    X_sin, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_con_tr = X_con.loc[X_sin_tr.index]
X_con_te = X_con.loc[X_sin_te.index]

print(f'Train: {X_sin_tr.shape} | fallecidos: {y_tr.sum():,} ({y_tr.mean()*100:.2f}%)')
print(f'Test : {X_sin_te.shape} | fallecidos: {y_te.sum():,} ({y_te.mean()*100:.2f}%)')

Train: (868557, 32) | fallecidos: 21,344 (2.46%)
Test : (217140, 32) | fallecidos: 5,336 (2.46%)


## 5. Modelado

### Estrategia de balanceo de clases

La clase positiva (fallecidos) representa ~2–3% de los datos. Se aplica:
- **SMOTE** en el pipeline de entrenamiento si `imbalanced-learn` está disponible
- **`class_weight='balanced'`** como fallback

Se comparan tres modelos:
1. Regresión Logística L2 (baseline)
2. Regresión Logística ElasticNet
3. Random Forest (referencia no-lineal)

Para cada modelo se entrena la versión **sin** y **con** `GRD_MORTALIDAD_SCORE`.

In [18]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def build_lr_pipeline(penalty='l2', solver='lbfgs', l1_ratio=None, use_smote=True):
    """Construir pipeline de LR con scaler y balanceo."""
    lr = LogisticRegression(
        penalty=penalty,
        solver=solver,
        l1_ratio=l1_ratio,
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight=None if (SMOTE_AVAILABLE and use_smote) else 'balanced'
    )
    steps = [('scaler', StandardScaler()), ('lr', lr)]
    if SMOTE_AVAILABLE and use_smote:
        return ImbPipeline([('smote', SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.1))] + steps)
    return Pipeline(steps)

print('Pipelines configurados correctamente.')

Pipelines configurados correctamente.


### 5.1 Regresión Logística L2 — versión sin data leakage

In [19]:
pipe_l2 = build_lr_pipeline(penalty='l2', solver='lbfgs')

param_grid_l2 = {'lr__C': [0.001, 0.01, 0.1, 1, 10, 100]}
if SMOTE_AVAILABLE:
    param_grid_l2 = {'lr__C': [0.001, 0.01, 0.1, 1, 10, 100]}

search_l2 = RandomizedSearchCV(
    pipe_l2, param_distributions=param_grid_l2, n_iter=6,
    scoring='roc_auc', cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
print('Entrenando LR L2 (sin leakage)...')
search_l2.fit(X_sin_tr, y_tr)
print(f'Mejor C: {search_l2.best_params_["lr__C"]} | ROC-AUC CV: {search_l2.best_score_:.4f}')

Entrenando LR L2 (sin leakage)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Mejor C: 0.001 | ROC-AUC CV: 0.9445


### 5.2 Regresión Logística ElasticNet — versión sin data leakage

In [20]:
pipe_en = build_lr_pipeline(penalty='elasticnet', solver='saga', l1_ratio=0.5)

param_dist_en = {
    'lr__C': [0.001, 0.01, 0.1, 1, 10],
    'lr__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

search_en = RandomizedSearchCV(
    pipe_en, param_distributions=param_dist_en, n_iter=15,
    scoring='roc_auc', cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
print('Entrenando LR ElasticNet (sin leakage)...')
search_en.fit(X_sin_tr, y_tr)
print(f'Mejores params: {search_en.best_params_} | ROC-AUC CV: {search_en.best_score_:.4f}')

Entrenando LR ElasticNet (sin leakage)...
Fitting 5 folds for each of 15 candidates, totalling 75 fits
Mejores params: {'lr__l1_ratio': 0.1, 'lr__C': 0.001} | ROC-AUC CV: 0.9445


### 5.3 Random Forest — versión sin data leakage

Se entrena un Random Forest como comparación no-lineal. Para datasets grandes puede ser lento;  
se usa `n_estimators` moderado y `max_samples` para acelerar.

In [21]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=50,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print('Entrenando Random Forest (sin leakage)...')
rf.fit(X_sin_tr, y_tr)
rf_auc_cv = cross_val_score(rf, X_sin_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
print(f'ROC-AUC CV (RF): {rf_auc_cv:.4f}')

Entrenando Random Forest (sin leakage)...
ROC-AUC CV (RF): 0.9557


### 5.4 Regresión Logística L2 — versión CON data leakage (referencia)

> ⚠️ **Advertencia:** Este modelo incluye `GRD_MORTALIDAD_SCORE`, que se asigna al alta.
> Sus métricas son infladas artificialmente y **no representan capacidad predictiva real**.
> Se incluye únicamente para cuantificar el impacto del leakage.

In [22]:
pipe_l2_leak = build_lr_pipeline(penalty='l2', solver='lbfgs')
param_grid_l2_leak = {'lr__C': [0.001, 0.01, 0.1, 1, 10, 100]}

search_l2_leak = RandomizedSearchCV(
    pipe_l2_leak, param_distributions=param_grid_l2_leak, n_iter=6,
    scoring='roc_auc', cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
print('Entrenando LR L2 (CON leakage)...')
search_l2_leak.fit(X_con_tr, y_tr)
print(f'Mejor C: {search_l2_leak.best_params_["lr__C"]} | ROC-AUC CV: {search_l2_leak.best_score_:.4f}')
print('(AUC inflado por data leakage)')

Entrenando LR L2 (CON leakage)...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Mejor C: 0.001 | ROC-AUC CV: 0.9483
(AUC inflado por data leakage)


## 6. Evaluación en test set

Se evalúan todos los modelos sobre el set de prueba.  
Dado el fuerte desbalance de clases, la **curva Precision-Recall y Average Precision (AP)**  
son métricas más informativas que ROC-AUC.

In [23]:
def evaluar_modelo(modelo, X_test, y_test, nombre, umbral=0.5):
    y_prob = modelo.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= umbral).astype(int)
    auc    = roc_auc_score(y_test, y_prob)
    ap     = average_precision_score(y_test, y_prob)
    print(f'\n{"=" * 60}')
    print(f'Modelo: {nombre}')
    print(f'ROC-AUC : {auc:.4f}  |  Avg Precision: {ap:.4f}')
    print(classification_report(y_test, y_pred, target_names=['VIVO', 'FALLECIDO'], digits=4))
    return y_prob, auc, ap

prob_l2,   auc_l2,   ap_l2   = evaluar_modelo(search_l2,      X_sin_te, y_te, 'LR L2 (sin leakage)')
prob_en,   auc_en,   ap_en   = evaluar_modelo(search_en,      X_sin_te, y_te, 'LR ElasticNet (sin leakage)')
prob_rf,   auc_rf,   ap_rf   = evaluar_modelo(rf,             X_sin_te, y_te, 'Random Forest (sin leakage)')
prob_leak, auc_leak, ap_leak = evaluar_modelo(search_l2_leak, X_con_te, y_te, 'LR L2 (CON leakage — referencia)')


Modelo: LR L2 (sin leakage)
ROC-AUC : 0.9446  |  Avg Precision: 0.2818
              precision    recall  f1-score   support

        VIVO     0.9978    0.8481    0.9169    211804
   FALLECIDO     0.1329    0.9241    0.2324      5336

    accuracy                         0.8500    217140
   macro avg     0.5653    0.8861    0.5746    217140
weighted avg     0.9765    0.8500    0.9000    217140


Modelo: LR ElasticNet (sin leakage)
ROC-AUC : 0.9446  |  Avg Precision: 0.2818
              precision    recall  f1-score   support

        VIVO     0.9978    0.8482    0.9169    211804
   FALLECIDO     0.1330    0.9243    0.2326      5336

    accuracy                         0.8501    217140
   macro avg     0.5654    0.8863    0.5748    217140
weighted avg     0.9765    0.8501    0.9001    217140


Modelo: Random Forest (sin leakage)
ROC-AUC : 0.9557  |  Avg Precision: 0.3971
              precision    recall  f1-score   support

        VIVO     0.9980    0.8665    0.9276    211804
   FA

### 6.1 Resumen comparativo

In [24]:
resumen = pd.DataFrame({
    'Modelo': ['LR L2 (sin leakage)', 'LR ElasticNet (sin leakage)', 'Random Forest (sin leakage)', 'LR L2 (CON leakage)'],
    'ROC-AUC': [auc_l2, auc_en, auc_rf, auc_leak],
    'Avg Precision': [ap_l2, ap_en, ap_rf, ap_leak],
})
resumen = resumen.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print(resumen.to_string(index=False))

                     Modelo  ROC-AUC  Avg Precision
Random Forest (sin leakage) 0.955686       0.397121
        LR L2 (CON leakage) 0.948793       0.293712
        LR L2 (sin leakage) 0.944582       0.281836
LR ElasticNet (sin leakage) 0.944581       0.281827


### 6.2 Curvas ROC y Precision-Recall

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelos_plot = [
    ('LR L2',         prob_l2,   auc_l2,   ap_l2),
    ('LR ElasticNet', prob_en,   auc_en,   ap_en),
    ('Random Forest', prob_rf,   auc_rf,   ap_rf),
    ('LR L2 (leak)',  prob_leak, auc_leak, ap_leak),
]

# ROC
for nombre, prob, auc, _ in modelos_plot:
    fpr, tpr, _ = roc_curve(y_te, prob)
    label = f'{nombre} (AUC={auc:.3f})'
    ls = '--' if 'leak' in nombre.lower() else '-'
    axes[0].plot(fpr, tpr, lw=1.8, linestyle=ls, label=label)
axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curva ROC')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Precision-Recall
baseline_pr = y_te.mean()
for nombre, prob, _, ap in modelos_plot:
    prec, rec, _ = precision_recall_curve(y_te, prob)
    label = f'{nombre} (AP={ap:.3f})'
    ls = '--' if 'leak' in nombre.lower() else '-'
    axes[1].plot(rec, prec, lw=1.8, linestyle=ls, label=label)
axes[1].axhline(y=baseline_pr, color='k', linestyle='--', lw=0.8, label=f'Baseline ({baseline_pr:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en plots/roc_pr_curves.png')

Gráfico guardado en plots/roc_pr_curves.png


### 6.3 Matriz de confusión — modelo final (LR L2 sin leakage)

Se usa el umbral de decisión por defecto (0.5). Para aplicaciones clínicas,  
es recomendable ajustar el umbral según el costo relativo de falsos negativos vs. falsos positivos.

In [26]:
y_pred_final = (prob_l2 >= 0.5).astype(int)
cm = confusion_matrix(y_te, y_pred_final)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['VIVO', 'FALLECIDO'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusión — LR L2 (sin leakage, umbral=0.5)')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en plots/confusion_matrix.png')

Gráfico guardado en plots/confusion_matrix.png


## 7. Feature Importance / Coeficientes

Para la regresión logística, los coeficientes estandarizados reflejan la contribución relativa  
de cada variable (dado que los datos fueron escalados con StandardScaler).  
Valores positivos aumentan el riesgo de mortalidad; negativos lo reducen.

In [27]:
# Extraer coeficientes del mejor modelo L2
best_lr = search_l2.best_estimator_
lr_step = best_lr.named_steps['lr']
feature_names = list(X_sin_tr.columns)
coefs = lr_step.coef_[0]

df_coefs = pd.DataFrame({'Feature': feature_names, 'Coeficiente': coefs})
df_coefs['Abs'] = df_coefs['Coeficiente'].abs()
df_coefs = df_coefs.sort_values('Abs', ascending=False).reset_index(drop=True)

print('Top 20 features por magnitud de coeficiente:')
print(df_coefs.head(20).to_string(index=False))

Top 20 features por magnitud de coeficiente:
                      Feature  Coeficiente      Abs
                GRD_SEVERIDAD     1.262315 1.262315
                         EDAD     0.903759 0.903759
        TIPO_INGRESO_URGENCIA     0.483133 0.483133
             N_PROCEDIMIENTOS     0.426293 0.426293
      TIPO_INGRESO_OBSTETRICA    -0.379433 0.379433
             ESP_NEONATOLOGÍA     0.294595 0.294595
                     ESP_OTRA     0.289881 0.289881
         ESP_MEDICINA INTERNA     0.264133 0.264133
                     GRD_PESO     0.261641 0.261641
               N_DIAGNOSTICOS     0.251033 0.251033
             ESP_OFTALMOLOGÍA    -0.234937 0.234937
      TIPO_INGRESO_PROGRAMADA    -0.234243 0.234243
ESP_TRAUMATOLOGÍA Y ORTOPEDIA    -0.225084 0.225084
         N_TRASLADOS_INTERNOS    -0.169598 0.169598
       ESP_PSIQUIATRÍA ADULTO    -0.152588 0.152588
       ESP_CIRUGÍA PEDIÁTRICA    -0.150935 0.150935
ESP_OBSTETRICIA Y GINECOLOGÍA    -0.129772 0.129772
                 ES

In [28]:
# Gráfico de coeficientes (top 20)
top_n = 20
df_plot = df_coefs.head(top_n).sort_values('Coeficiente')

colors = ['#d62728' if c > 0 else '#1f77b4' for c in df_plot['Coeficiente']]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(df_plot['Feature'], df_plot['Coeficiente'], color=colors, edgecolor='none', height=0.7)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente estandarizado (log-odds)')
ax.set_title(f'Top {top_n} features — LR L2 sin leakage')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'coeficientes_lr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en plots/coeficientes_lr.png')

Gráfico guardado en plots/coeficientes_lr.png


In [29]:
# Feature importance Random Forest (top 20)
imp_rf = pd.DataFrame({'Feature': feature_names, 'Importancia': rf.feature_importances_})
imp_rf = imp_rf.sort_values('Importancia', ascending=False).head(top_n)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp_rf['Feature'].iloc[::-1], imp_rf['Importancia'].iloc[::-1], color='#2ca02c', edgecolor='none')
ax.set_xlabel('Importancia (impureza media)')
ax.set_title(f'Top {top_n} features — Random Forest')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'importancia_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en plots/importancia_rf.png')

Gráfico guardado en plots/importancia_rf.png


### Interpretación clínica

Variables con mayor peso positivo (aumentan riesgo de mortalidad):
- **EDAD**: mayor edad → mayor mortalidad esperada
- **GRD_SEVERIDAD**: pacientes con mayor severidad clínica tienen peor pronóstico
- **N_DIAGNOSTICOS / N_PROCEDIMIENTOS**: mayor complejidad del caso
- **DIAS_HOSPITALIZACION**: estancias largas pueden reflejar complicaciones post-ingreso

Variables con peso negativo (protectoras o asociadas a menor mortalidad):
- **SEXO_BIN (MUJER)**: consistente con menor mortalidad hospitalaria observada en mujeres
- Ciertas especialidades o tipos de ingreso pueden reflejar procesos electivos de bajo riesgo

> **Nota:** Los coeficientes son asociaciones, no efectos causales.  
> El modelo capta patrones estadísticos del dataset; la interpretación requiere contexto clínico.

## 8. Exportación de resultados

In [30]:
# Guardar modelo final (LR L2 sin leakage)
model_path = MODELS_DIR / 'modelo_mortalidad.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({
        'model': search_l2.best_estimator_,
        'features': ALL_FEATURES_SIN,
        'best_params': search_l2.best_params_,
        'roc_auc_test': auc_l2,
        'avg_precision_test': ap_l2,
    }, f)
print(f'Modelo guardado en {model_path}')

# Guardar coeficientes
coef_path = DATA_PROC_DIR / 'coeficientes_mortalidad.csv'
df_coefs.drop(columns='Abs').to_csv(coef_path, index=False)
print(f'Coeficientes guardados en {coef_path}')

# Guardar resumen de modelos
resumen_path = DATA_PROC_DIR / 'resumen_modelos_mortalidad.csv'
resumen.to_csv(resumen_path, index=False)
print(f'Resumen guardado en {resumen_path}')

Modelo guardado en ..\models\modelo_mortalidad.pkl
Coeficientes guardados en ..\data\processed\coeficientes_mortalidad.csv
Resumen guardado en ..\data\processed\resumen_modelos_mortalidad.csv


## 9. Resumen metodológico

| Aspecto | Decisión |
|---|---|
| Variable objetivo | Binaria: TIPOALTA == 'FALLECIDO' |
| Split | 80/20 estratificado |
| Balanceo | SMOTE (sampling_strategy=0.1) o class_weight='balanced' |
| Validación cruzada | StratifiedKFold 5-fold |
| Métrica principal | ROC-AUC + Avg Precision (PR-AUC) |
| Data leakage | GRD_MORTALIDAD_SCORE excluido del modelo final; incluido en versión de referencia |
| Encoding hospital | Target Encoding o Frequency Encoding |
| Modelo seleccionado | LR L2 sin leakage (interpretable, generalizable) |

### Limitaciones
1. **GRD se asigna al alta**: severidad, peso y mortalidad GRD son retrospectivos. En un sistema de alerta temprana real, estos campos no estarían disponibles al ingreso.
2. **Diagnósticos y procedimientos**: la cantidad (N_DIAGNOSTICOS) es disponible en tiempo real, pero no qué diagnóstico específico.
3. **Leakage temporal**: días de hospitalización también puede ser parcialmente retrospectivo según el momento de predicción deseado.
4. **Heterogeneidad entre hospitales**: el target encoding puede capturar diferencias de codificación más que diferencias reales en riesgo.

### Próximos pasos
- Calibración del modelo (Platt scaling / isotonic regression)
- Ajuste del umbral de decisión según sensibilidad clínica requerida
- Validación externa en datos de otro año
- Análisis de equidad (fairness) por sexo, previsión y región